In [0]:
data=[("1","Pallavi"),("2","Richa"),("3","Neha")]
columns=[("Student_id"),("Student_name")]

df1=spark.createDataFrame(data,columns)
df1.createOrReplaceTempView("df1") 
display(df1)

In [0]:
data=[("1","English","70"),("1","Hindi","40"),("1","Math","20"),("2","English","30"),("2","Hindia","90"),("2","Math","80"),("3","English","60"),("3","Hindi","10")]
columns=[("Student_id"),("Subject_name"),("Marks")]

df2=spark.createDataFrame(data,columns)
df2.createOrReplaceTempView("df2") 
display(df2)

Alternative to SQL

In [0]:
df5 = df1.join(df2,df1.Student_id==df2.Student_id).drop(df2.Student_id)
display(df5)

In [0]:
from pyspark.sql.functions import *

df_per = df5.groupBy('Student_id','Student_name').agg((sum('Marks')/count('*')).alias('Percentage'))
display(df_per)

In [0]:
df_final = df_per.select('*',
    when(df_per.Percentage >= 70, 'Distinction')
    .when((df_per.Percentage < 70) & (df_per.Percentage > 60), 'First')
    .when((df_per.Percentage <= 60) & (df_per.Percentage > 50), 'Second')
    .when((df_per.Percentage <= 50) & (df_per.Percentage > 40), 'Third')
    .when(df_per.Percentage <= 40, 'Fail')
    .otherwise('Unknown')
    .alias('Result')
)
display(df_final)

In [0]:
df3 = spark.sql("""
SELECT df1.Student_id, df1.Student_name, df2.Subject_name, df2.Marks
FROM df1
INNER JOIN df2 ON df1.Student_id = df2.Student_id""")

display(df3)


In [0]:
df4 = spark.sql("""SELECT df3.Student_id, df3.Student_name, df3.Subject_name,(df3.Marks / 100) * 100 AS Percentage,
CASE
  WHEN df3.Marks >= 70 THEN 'Distinction'
  WHEN df3.Marks >= 40 THEN 'Pass'
  ELSE 'Fail'
  END AS Result
from df3""") 
display(df4)
